# Phase 6 — Prediction-Point Review & Leakage Audit
**Topic:** Predicting Road Traffic Collision Severity and Identifying High-Risk Locations and Times  
**Dataset:** UK DfT STATS19 Road Safety Data  
**Pipeline Tasks:** Tasks 17, 18, and 19  
**Deliverables:** Formal Prediction-Point Statement, Leakage Audit Report, Leakage-Safe Feature Set (`master_dataset_v3_leakage_safe.csv`)

---
### Objective
Define exactly what the Road Safety AI model is allowed to know at the moment of prediction and prevent future, downstream, or target-proxy information from entering model features.

## Task 17 — Define Prediction Point

### 17.1 Application Scenario & Prediction Moment
The Road Safety AI System is deployed for **prospective decision-support**:
1. **Pre-collision / Real-time Risk Scoring:** Evaluating hazard levels of specific road segments, times, and environmental conditions to support police patrol allocation and highway safety interventions.
2. **At-collision Dispatch Planning:** When a collision is first reported (e.g., via emergency call with known vehicle types and road conditions), estimating severity to guide emergency medical resource allocation before on-scene medical assessments are finalized.

### 17.2 Permissible vs. Prohibited Information
- **Permissible Inputs (Available Before or At Collision):**
  - Environmental context: `weather_conditions`, `light_conditions`, `road_surface_conditions`, `special_conditions_at_site`.
  - Road infrastructure: `road_type`, `speed_limit`, `junction_detail`, `junction_control`, `urban_or_rural_area`, `first_road_class`.
  - Temporal context: `day_of_week`, `is_weekend`, `month`, `season`, `hour_of_day`, `time_of_day_bucket`.
  - Vehicle context: `number_of_vehicles`, vehicle type presence flags (`has_car`, `has_motorcycle`, `has_hgv`, etc.).
  - Spatial coordinates: `latitude`, `longitude` (for geographic analysis/hotspots).
- **Prohibited Information (Post-Collision / Downstream Outcomes):**
  - Individual casualty medical assessments: `worst_casualty_severity`, `casualty_class`, `injury_based`.
  - Demographic casualty flags: `has_child_casualty`, `has_passenger_casualty`, `has_elderly_casualty`, `pedestrian_involved`.
  - Administrative / Police outcome flags: `did_police_officer_attend_scene_of_accident`, `enhanced_severity_collision`.
  - Target variables themselves: `collision_severity` (classification target), `number_of_casualties` (regression target).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

master_df = pd.read_csv(data_path / "master_dataset_v2_engineered.csv", low_memory=False)
inventory_df = pd.read_csv(data_path / "feature_inventory.csv")

print(f"Loaded Master Dataset v2 shape: {master_df.shape}")
print(f"Loaded Feature Inventory entries: {len(inventory_df)}")
print("Sample columns:", master_df.columns[:8].tolist())

Loaded Master Dataset v2 shape: (513801, 65)
Loaded Feature Inventory entries: 43
Sample columns: ['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force']


## Task 18 — Leakage Audit

### 18.1 Comprehensive Leakage Taxonomy
We audit every field across four distinct leakage categories:
1. **Direct Target Leakage:** Target variables included in predictors ($X$).
2. **Outcome / Downstream Leakage:** Information determined after the collision occurs (e.g. casualty injuries).
3. **Proxy / Target-Near-Duplicate Leakage:** Fields derived directly from the target (e.g. `enhanced_severity_collision`, `collision_adjusted_severity_serious`).
4. **Administrative / Investigation Leakage:** Post-event procedural details (e.g. `did_police_officer_attend_scene_of_accident`).

In [2]:
# Audit every column in master_df
leakage_audit = []

for col in master_df.columns:
    col_lower = col.lower()
    if col in ["collision_severity", "number_of_casualties"]:
        leakage_audit.append({"column": col, "leakage_type": "Direct Target", "action": "EXCLUDE from X", "reason": "Target variable"})
    elif any(term in col_lower for term in ["enhanced", "adjusted", "injury_based"]):
        leakage_audit.append({"column": col, "leakage_type": "Target Proxy", "action": "EXCLUDE from X", "reason": "Post-event calculated severity proxy"})
    elif col in ["worst_casualty_severity", "pedestrian_involved", "has_child_casualty", "has_passenger_casualty", "has_elderly_casualty"]:
        leakage_audit.append({"column": col, "leakage_type": "Downstream Casualty Outcome", "action": "EXCLUDE from prospective model", "reason": "Casualty table outcome evaluated after collision"})
    elif col in ["did_police_officer_attend_scene_of_accident"]:
        leakage_audit.append({"column": col, "leakage_type": "Administrative / Post-event", "action": "EXCLUDE from X", "reason": "Police attendance occurs post-event"})
    elif col in ["collision_index", "collision_ref_no"]:
        leakage_audit.append({"column": col, "leakage_type": "Identifier", "action": "REFERENCE ONLY", "reason": "Arbitrary tracking key, not predictive"})
    else:
        leakage_audit.append({"column": col, "leakage_type": "None (Valid Pre-crash Feature)", "action": "RETAIN as candidate", "reason": "Known prior to or at collision"})

leakage_report_df = pd.DataFrame(leakage_audit)
print("Leakage Audit Summary:")
print(leakage_report_df["action"].value_counts())
leakage_report_df[leakage_report_df["action"] != "RETAIN as candidate"]

Leakage Audit Summary:
action
RETAIN as candidate               51
EXCLUDE from X                     7
EXCLUDE from prospective model     5
REFERENCE ONLY                     2
Name: count, dtype: int64


,column,leakage_type,action,reason
0,collision_index,Identifier,REFERENCE ONLY,"Arbitrary tracking key, not predictive"
2,collision_ref_no,Identifier,REFERENCE ONLY,"Arbitrary tracking key, not predictive"
8,collision_severity,Direct Target,EXCLUDE from X,Target variable
10,number_of_casualties,Direct Target,EXCLUDE from X,Target variable
35,did_police_officer_attend_scene_of_accident,Administrative / Post-event,EXCLUDE from X,Police attendance occurs post-event
38,enhanced_severity_collision,Target Proxy,EXCLUDE from X,Post-event calculated severity proxy
39,collision_injury_based,Target Proxy,EXCLUDE from X,Post-event calculated severity proxy
40,collision_adjusted_severity_serious,Target Proxy,EXCLUDE from X,Post-event calculated severity proxy
41,collision_adjusted_severity_slight,Target Proxy,EXCLUDE from X,Post-event calculated severity proxy
54,pedestrian_involved,Downstream Casualty Outcome,EXCLUDE from prospective model,Casualty table outcome evaluated after collision


## Task 19 — Remove Inappropriate Predictive Features & Build Leakage-Safe Dataset

### 19.1 Constructing the Leakage-Safe Feature Set
We construct `master_dataset_v3_leakage_safe.csv` by retaining:
1. Target variables (`collision_severity`, `number_of_casualties`) clearly marked.
2. Join index (`collision_index`) for reference.
3. Pre-crash environmental, road infrastructure, vehicle-type, and temporal features.
4. Downstream casualty outcomes are segregated into an `analysis_only` subset for EDA and sanity checks, preventing them from contaminating the prospective model.

In [3]:
# Prohibited columns to exclude from prospective model predictors
exclude_cols = leakage_report_df[leakage_report_df["action"].isin(["EXCLUDE from X", "EXCLUDE from prospective model"])]["column"].tolist()
# Filter out targets so targets stay in master table for Phase 8/9 splitting
model_exclude_cols = [c for c in exclude_cols if c not in ["collision_severity", "number_of_casualties"]]

print(f"Total columns excluded from prospective feature set: {len(model_exclude_cols)}")
print("Excluded columns:", model_exclude_cols)

# Retain casualty outcome variables separately in a dedicated EDA table
casualty_eda_cols = ["collision_index", "collision_severity", "number_of_casualties"] + [c for c in model_exclude_cols if c in master_df.columns and "casualty" in c or "pedestrian" in c]
casualty_eda_df = master_df[[c for c in casualty_eda_cols if c in master_df.columns]].copy()
casualty_eda_df.to_csv(data_path / "casualty_outcomes_eda_only.csv", index=False)
print("Saved: casualty_outcomes_eda_only.csv for downstream descriptive analysis")

# Drop non-prospective columns for v3 master dataset
clean_v3_df = master_df.drop(columns=[c for c in model_exclude_cols if c in master_df.columns])
print(f"Master Dataset v3 (Leakage-Safe) shape: {clean_v3_df.shape}")

# Save Phase 6 Outputs
clean_v3_df.to_csv(data_path / "master_dataset_v3_leakage_safe.csv", index=False)
leakage_report_df.to_csv(data_path / "leakage_audit_report.csv", index=False)

print("\n" + "="*70)
print("PHASE 6 COMPLETE: Leakage-Safe Master Dataset v3 Saved Successfully!")
print("="*70)

Total columns excluded from prospective feature set: 10
Excluded columns: ['did_police_officer_attend_scene_of_accident', 'enhanced_severity_collision', 'collision_injury_based', 'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight', 'pedestrian_involved', 'has_passenger_casualty', 'has_child_casualty', 'has_elderly_casualty', 'worst_casualty_severity']
Saved: casualty_outcomes_eda_only.csv for downstream descriptive analysis
Master Dataset v3 (Leakage-Safe) shape: (513801, 55)

PHASE 6 COMPLETE: Leakage-Safe Master Dataset v3 Saved Successfully!
